# DSFB-Database — Reproduction Notebook

This notebook runs **dsfb-database** from scratch on **real public datasets**, end-to-end, in Colab (or any Jupyter kernel). No synthetic substitutes are used on the real-data path. The only deterministic-synthetic section is the TPC-DS-shaped perturbation harness, whose entire purpose is ground-truth-validated F1/TTD measurement (§8 of the paper).

## Non-claims (read first)

1. DSFB-Database does not optimise queries, replace the query optimiser, or modify execution plans.
2. DSFB-Database does not claim causal correctness; motifs represent structural consistency given observed signals, not root causes.
3. DSFB-Database does not provide a forecasting or predictive guarantee; precursor structure is structural, not probabilistic.
4. DSFB-Database does not provide ground-truth-validated detection on real workloads; we evaluate via injected perturbations, plan-hash concordance, and replay determinism.
5. DSFB-Database does not claim a universal SQL grammar; motifs are engine-aware, telemetry-aware, and workload-aware.

## What this notebook shows

* **Real data, every run.** Four real public datasets are shipped as representative samples inside the crate at `examples/data/`:
    * `ceb_sample.csv` — 5,000 true/estimated PostgreSQL cardinality rows, sliced from the first 400 CEB-IMDb pickles (https://github.com/learnedsystems/CEB, MIT).
    * `job_trace_sample.csv` — EXPLAIN ANALYZE trace over all 113 JOB queries × 3 replays, produced by running `scripts/fetch_job.sh` against IMDb in DuckDB (https://github.com/gregrahn/join-order-benchmark + https://event.cwi.nl/da/job/imdb.tgz, MIT).
    * `queries.txt` — full 11,136-query SQLShare text release (https://uwescience.github.io/sqlshare/data_release.html, CC-BY 4.0). Analysed in `sqlshare-text` mode (workload-phase motif only, over ordinal-position buckets — the public release has no timing/user metadata).
    * `snowset_sample.csv` — first 5,000 real rows from the Snowset Cornell mirror (http://www.cs.cornell.edu/~midhul/snowset/snowset-main.csv.gz, CC-BY 4.0).
* **Controlled-perturbation pipeline** (TPC-DS-shaped, deterministic in-process) provides the only ground-truth-validated metrics (F1/TTD/FAR) in the paper.
* **Heavy-fetch path**, for users who want the full corpora (not required for this notebook), is documented in the final cell.
* Replay-determinism is pinned by fingerprint: the test suite verifies SHA-256 of the canonical stream + episode outputs.

## 1. Clean working dir, install Rust, clone repo

In [ ]:
import os, sys, shutil, subprocess

# Wipe any previous repo checkout so every run starts from scratch.
if os.path.isdir('dsfb'):
    shutil.rmtree('dsfb')

if shutil.which('cargo') is None:
    subprocess.check_call(['bash', '-lc',
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
        "sh -s -- -y --default-toolchain stable"])
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
subprocess.check_call(['cargo', '--version'])
subprocess.check_call(['git', 'clone', '--depth', '1',
                       'https://github.com/infinityabundance/dsfb.git'])
REPO  = os.path.abspath('dsfb')
CRATE = os.path.join(REPO, 'crates', 'dsfb-database')
OUT   = os.path.join(CRATE, 'out')
os.makedirs(OUT, exist_ok=True)
print('crate path:', CRATE)

## 2. Build + run the test suite

Includes the pinned-fingerprint replay-determinism tests and the non-claim lock.

In [ ]:
subprocess.check_call(['cargo', 'build', '--release'], cwd=CRATE)
subprocess.check_call(['cargo', 'test',  '--release'], cwd=CRATE)

## 3. Controlled-perturbation pipeline (TPC-DS-shaped, deterministic)

This is the only section with ground-truth windows. The paper's F1/TTD/FAR
numbers come from here, not from the real-data runs below — real-data
runs do not have labelled ground truth (see non-claim #4).

In [ ]:
subprocess.check_call(['./target/release/dsfb-database', 'reproduce',
                       '--seed', '42', '--out', OUT], cwd=CRATE)
subprocess.check_call(['./target/release/dsfb-database', 'replay-check',
                       '--seed', '42'], cwd=CRATE)
subprocess.check_call(['./target/release/dsfb-database', 'elasticity',
                       '--seed', '42', '--out', OUT], cwd=CRATE)

import pandas as pd
metrics = pd.read_csv(os.path.join(OUT, 'tpcds.metrics.csv'))
metrics

In [ ]:
from IPython.display import Image, display
for m in ['plan_regression_onset', 'cardinality_mismatch_regime',
         'contention_ramp', 'cache_collapse', 'workload_phase_transition']:
    path = os.path.join(OUT, f'tpcds.{m}.png')
    if os.path.exists(path):
        print(m)
        display(Image(path))

## 4. Real-data runs

Each of the four real public datasets is processed by its adapter
against the bundled sample. Outputs per dataset:

* `<dataset>.episodes.csv` — emitted episodes (motif, channel, t_start, t_end, ...).
* `<dataset>.<motif>.png`  — per-motif residual-overlay figures.
* `<dataset>.provenance.txt` — source file + non-claim block.

There are no ground-truth windows here (see non-claim #4). Real-data
runs demonstrate determinism + per-dataset residual grammar coverage,
not F1.

In [ ]:
def run_real(dataset, path):
    out = os.path.join(OUT, f'real_{dataset}')
    os.makedirs(out, exist_ok=True)
    cmd = ['./target/release/dsfb-database', 'run',
           '--dataset', dataset, '--path', path, '--out', out]
    print('$', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=CRATE)
    return out

real_outputs = {
    'ceb':           run_real('ceb',           'examples/data/ceb_sample.csv'),
    'job':           run_real('job',           'examples/data/job_trace_sample.csv'),
    'sqlshare-text': run_real('sqlshare-text', 'examples/data/queries.txt'),
    'snowset':       run_real('snowset',       'examples/data/snowset_sample.csv'),
}

In [ ]:
import glob
for ds, out in real_outputs.items():
    print('=' * 60)
    print('dataset:', ds)
    ep_csv = glob.glob(os.path.join(out, '*.episodes.csv'))
    if ep_csv:
        df = pd.read_csv(ep_csv[0])
        print(f'episodes: {len(df)} rows')
        display(df.head(10))
    for png in sorted(glob.glob(os.path.join(out, '*.png'))):
        print(os.path.basename(png))
        display(Image(png))

## 5. Download all artifacts as a single zip

Bundles every generated CSV, JSON, PNG, and provenance file under `out/` into a single `dsfb_database_artifacts.zip` next to the notebook. In Colab, the file is handed to `google.colab.files.download()` so you can grab it with one click; outside Colab, the zip path is printed so you can fetch it via the filesystem.

The zip is built from the real-data `out/real_*` directories plus the controlled-perturbation `out/tpcds.*` outputs; the `target/` build tree is excluded so the archive stays small (<10 MB on the bundled samples).

In [ ]:
import os, shutil, glob

# Build the zip from the crate's out/ tree. We archive into the notebook
# working directory (not into CRATE) so the file is visible to the Colab
# file browser and the local filesystem alike.
artifact_root = '/content/dsfb_database_artifacts' if os.path.isdir('/content') else os.path.abspath('dsfb_database_artifacts')
if os.path.isdir(artifact_root):
    shutil.rmtree(artifact_root)
shutil.copytree(OUT, artifact_root)

# Include fingerprints + non-claim block so the archive is self-describing.
with open(os.path.join(artifact_root, 'README.txt'), 'w') as f:
    f.write('DSFB-Database reproduction archive\n')
    f.write('==================================\n\n')
    f.write('Produced by colab/dsfb_database_repro.ipynb.\n\n')
    f.write('Contents:\n')
    f.write('  tpcds.*       controlled-perturbation pipeline (F1/TTD/FAR)\n')
    f.write('  real_<ds>/    real-data adapter outputs (episodes + figures)\n')
    f.write('  provenance.txt / *.provenance.txt  per-stream non-claim block + source\n\n')
    f.write('Fingerprints are pinned by the test suite (cargo test --release).\n')

zip_base = os.path.abspath('dsfb_database_artifacts')
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=artifact_root)
print('artifact zip ready:', zip_path, f'({os.path.getsize(zip_path)/1e6:.2f} MB)')

# One-click download inside Colab; graceful fallback elsewhere.
try:
    from google.colab import files as _files  # type: ignore
    _files.download(zip_path)
except Exception as e:
    print('(not in Colab: download manually from', zip_path, ')')
    print('  reason:', e)


## 6. (Optional) Full-corpus fetch paths

The bundled samples above are **real data** (sliced directly from the
authoritative public dumps), large enough to exercise every motif the
dataset can support. If you want to process the **full** public corpus
of each dataset — e.g. for an independent re-analysis — the fetch
scripts under `scripts/` pull the authoritative artefacts and write the
same CSV schema. They are heavy (Snowset is ~7.5 GB gzipped; JOB
requires downloading the 1.17 GB IMDb dump and loading it into DuckDB;
CEB downloads a ~1 GB pickle tarball from Dropbox) and are
**not required** to reproduce the paper's figures or any of the
fingerprints pinned by the test suite.

```bash
cd crates/dsfb-database
bash scripts/fetch_ceb.sh             # CEB pickles -> data/ceb_subset.csv      (~50 MB)
bash scripts/fetch_job.sh             # IMDb 1.17 GB + DuckDB load + 113 x N EXPLAIN ANALYZE
bash scripts/fetch_snowset_subset.sh  # Snowset 7.5 GB dump -> data/snowset_shard.csv
bash scripts/fetch_sqlshare.sh        # SQLShare queries.txt (already bundled in examples/data/)
bash scripts/build_tpcds.sh           # in-process; no download required

./target/release/dsfb-database run --dataset ceb           --path data/ceb_subset.csv
./target/release/dsfb-database run --dataset job           --path data/job_trace.csv
./target/release/dsfb-database run --dataset snowset       --path data/snowset_shard.csv
./target/release/dsfb-database run --dataset sqlshare-text --path examples/data/queries.txt
```

Every output stream is fingerprinted; re-running any of these commands
on the same input is bytewise-deterministic.